# 06. Random Forest Classifier - Rheumatoid Arthritis Classification

## Model Information & Ownership
- **Model Architecture**: Random Forest Classifier (Ensemble Bagging of Decision Trees)
- **Assigned Owner**: **Teammate 2 (Nakshtr Gupta)**
- **Target Export Artifact**: `models/random_forest/random_forest_pipeline.joblib`
- **Scientific Goal**: Capture non-linear interactions and assess Gini / permutation feature importance without strict linearity assumptions.

---

## Shared Input Interface & Contract
Downstream models must consume the standardized **14 candidate clinical features** loaded from the reproducible stratified split in `data/splits/`.

- **Input Features (14)**:
  - Demographics (2): `Age`, `Gender`
  - Numeric Labs (6): `ESR`, `CRP`, `RF`, `Anti-CCP`, `C3`, `C4`
  - Categorical Labs (6): `HLA-B27`, `ANA`, `Anti-Ro`, `Anti-La`, `Anti-dsDNA`, `Anti-Sm`
- **Excluded Columns**: `Disease` (prevents leakage) and `patient_row_id` (tracking key).
- **Target Vector (`y`)**: `Target_RA` (1 = Rheumatoid Arthritis, 0 = Non-RA).

> **CRITICAL RULE**: Do not fit imputers, scalers, or encoders on `X_test`. All preprocessing must be fitted strictly on `X_train` (e.g., using a `Pipeline`).


In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
root = Path("..").resolve() if Path("..").resolve().name == "RA_Project" else Path(".").resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_loader import (
    load_train_data,
    load_test_data,
    get_feature_names,
    get_feature_metadata,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET_COLUMN
)

# Load reproducible stratified train and test partitions
X_train, y_train = load_train_data(return_X_y=True)
X_test, y_test = load_test_data(return_X_y=True)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape} (Positive RA cases: {y_train.sum()} [{y_train.mean():.2%}])")
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape} (Positive RA cases: {y_test.sum()} [{y_test.mean():.2%}])")
print(f"\nStandardized Features ({len(get_feature_names())}):\n{get_feature_names()}")


In [ ]:
# Verify data contract
assert X_train.shape[1] == 14, "X_train must have exactly 14 features"
assert X_test.shape[1] == 14, "X_test must have exactly 14 features"
assert "Disease" not in X_train.columns, "Target leakage detected: 'Disease' in X_train!"
assert "Target_RA" not in X_train.columns, "Target leakage detected: 'Target_RA' in X_train!"
assert set(y_train.unique()).issubset({0, 1}), "y_train must be strictly binary (0 or 1)"
print("Data loading interface verified successfully.")


## TODO 1: Define Preprocessing Pipeline
- Set up `ColumnTransformer` for numeric features (`NUMERIC_FEATURES`) and categorical features (`CATEGORICAL_FEATURES`).
- Handle missing values using an appropriate strategy (e.g., median for numeric, missing-indicator or constant category for categorical).
- Apply scaling or encoding as required by the algorithm.
- Ensure the pipeline can be fitted on `X_train` and applied to `X_test` without data leakage.


In [ ]:
# TODO: Implement preprocessing pipeline
# Example:
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import StandardScaler, OneHotEncoder


## TODO 2: Hyperparameter Tuning via Stratified 5-Fold Cross-Validation
- Use `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` on `X_train`.
- Optimize hyperparameters using `GridSearchCV` or `RandomizedSearchCV`.
- Track evaluation metrics: ROC-AUC, PR-AUC, F1-Score, and Sensitivity.


In [ ]:
# TODO: Implement Stratified 5-Fold Cross-Validation and hyperparameter search on X_train
# from sklearn.model_selection import StratifiedKFold, GridSearchCV


## TODO 3: Final Training on Full X_train and Unbiased Test Evaluation
- Fit the best pipeline on the entire `X_train` dataset.
- Evaluate the model once on `X_test`.
- Report test ROC-AUC, PR-AUC, confusion matrix, precision, recall, and specificity.
- Inspect feature importances / model coefficients.


In [ ]:
# TODO: Evaluate best model on X_test
# from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix


## TODO 4: Export Saved Model Pipeline for Dashboard Integration
- Export the fitted pipeline to `models/random_forest/random_forest_pipeline.joblib`.
- Test loading the saved artifact and predicting on a sample record.
- Verify that the saved artifact does not require retraining when loaded.


In [ ]:
# TODO: Save trained model pipeline
# import joblib
# joblib.dump(fitted_pipeline, "../models/random_forest/random_forest_pipeline.joblib")
